# 🎨 AUTOMATIC1111 WebUI (Lobe Theme + Bộ Trợ lý Prompt Toàn diện)

Notebook triển khai **AUTOMATIC1111 WebUI** với giao diện **Lobe Theme** cao cấp & trợ lý gợi ý từ khóa prompt:
- 💎 **Giao diện Lobe Theme**: Hiện đại phong cách Midjourney / Apple, Dark mode, Glassmorphism, tối giản và chuyên nghiệp.
- 🧠 **Trợ lý Prompt All-In-One (`sd-webui-prompt-all-in-one`)**:
  - 🇻🇳 **Dịch trực tiếp tiếng Việt -> tiếng Anh** ngay trong WebUI, không cần mở Google Dịch.
  - 📚 **Từ điển từ khóa phân loại sẵn**: Khuôn mặt, Tóc, Áo quần, Tư thế, Góc chụp, Ánh sáng, Chất lượng... chỉ cần click chọn.
  - 🎴 **Tự động nhận diện LoRA**: Hiển thị danh sách LoRA có sẵn để bấm 1-click chèn vào prompt kèm thanh chỉnh weight.
- ⚡ **Thư viện Style Mẫu (`styles.csv`)**: Tích hợp sẵn hàng loạt style (Krea 2 Succubus, Realistic Vision, SDXL Lightning, Chân dung nghệ thuật...).
- 💾 **Dùng chung kho Model với ComfyStudio trên Google Drive** (`MyDrive/ComfyStudio/models`): Không cần tải lại.
- 🚀 **Download đa luồng `aria2c`**: Bypass lỗi 403 Civitai, tải tốc độ tối đa.


In [ ]:
#@title 1. Kết nối Google Drive & Cấu hình Thư mục Dùng chung
import os
import shutil
from google.colab import drive

print("🚀 Đang kết nối Google Drive...")
drive.mount('/content/drive')

# Thư mục gốc lưu trữ model trên Drive (dùng chung với ComfyStudio)
DRIVE_BASE = "/content/drive/MyDrive/ComfyStudio/models"
DRIVE_OUTPUTS = "/content/drive/MyDrive/ComfyStudio/outputs/automatic1111"

# Các thư mục model cần thiết cho Automatic1111
FOLDERS = [
    "checkpoints",
    "loras",
    "vae",
    "controlnet",
    "upscale_models",
    "embeddings"
]

for f in FOLDERS:
    os.makedirs(os.path.join(DRIVE_BASE, f), exist_ok=True)
os.makedirs(DRIVE_OUTPUTS, exist_ok=True)

print(f"✅ Đã sẵn sàng thư mục Models trên Drive: {DRIVE_BASE}")
print(f"📁 Thư mục xuất ảnh tự động: {DRIVE_OUTPUTS}")


In [ ]:
#@title 2. Cài đặt AUTOMATIC1111 + Lobe Theme + Bộ Trợ lý Prompt All-In-One
import os
import shutil
import subprocess

WEBUI_DIR = "/content/stable-diffusion-webui"
EXT_DIR = "/content/stable-diffusion-webui/extensions"

# 1. Clone Automatic1111 WebUI
if not os.path.exists(WEBUI_DIR):
    print("📦 Đang clone AUTOMATIC1111 Stable Diffusion WebUI...")
    !git clone --depth 1 https://github.com/AUTOMATIC1111/stable-diffusion-webui.git /content/stable-diffusion-webui
else:
    print("ℹ️ Thư mục WebUI đã tồn tại, cập nhật code mới nhất...")
    %cd /content/stable-diffusion-webui
    !git pull

%cd /content/stable-diffusion-webui

# 2. Cài đặt aria2 và thư viện phụ trợ xformers
print("⚡ Cài đặt aria2 và xformers...")
!apt-get update -qq && apt-get install -y -qq aria2
!pip install -q xformers==0.0.28.post1 --index-url https://download.pytorch.org/whl/cu124

# 3. Cài đặt bộ Extensions & Giao diện Lobe Theme
os.makedirs(EXT_DIR, exist_ok=True)

# A. Giao diện Lobe Theme (Hiện đại phong cách Apple / Midjourney)
if not os.path.exists(f"{EXT_DIR}/sd-webui-lobe-theme"):
    print("💎 Đang cài đặt Giao diện Lobe Theme...")
    !git clone --depth 1 https://github.com/canisminor1990/sd-webui-lobe-theme.git {EXT_DIR}/sd-webui-lobe-theme

# B. Trợ lý Prompt All-In-One (Dịch tiếng Việt, Từ điển từ khóa phân loại, Quản lý LoRA)
if not os.path.exists(f"{EXT_DIR}/sd-webui-prompt-all-in-one"):
    print("🧠 Đang cài đặt Trợ lý Prompt All-In-One (Dịch thuật + Từ điển tag)...")
    !git clone --depth 1 https://github.com/Physton/sd-webui-prompt-all-in-one.git {EXT_DIR}/sd-webui-prompt-all-in-one

# C. Tag Autocomplete (Tự động gợi ý từ khóa khi gõ)
if not os.path.exists(f"{EXT_DIR}/a1111-sd-webui-tagcomplete"):
    print("🔌 Đang cài đặt Extension Tag Autocomplete...")
    !git clone --depth 1 https://github.com/DominikDoom/a1111-sd-webui-tagcomplete.git {EXT_DIR}/a1111-sd-webui-tagcomplete

# D. ControlNet Extension
if not os.path.exists(f"{EXT_DIR}/sd-webui-controlnet"):
    print("🔌 Đang cài đặt Extension sd-webui-controlnet...")
    !git clone --depth 1 https://github.com/Mikubill/sd-webui-controlnet.git {EXT_DIR}/sd-webui-controlnet

# 4. Tạo sẵn bộ Style Mẫu (styles.csv) để nạp 1-click trong WebUI
styles_content = """name,prompt,negative_prompt
"🔥 Krea 2 / LUSTIFY Succubus","A woman is captured as a melancholic succubus with horns and wings, posed against a backdrop of ethereal white cherry blossoms and cool blue atmospheric lighting, very realistic, 8k, cinematic photography, <lora:KNP_000003000:1.0>","worst quality, low quality, bad anatomy, deformed, distorted, blurry, cartoon, 3d render"
"📷 Realistic Vision V6 (Paris Candid)","RAW street style photography of a beautiful model in Paris, natural candid smile, stylish beige trench coat, soft golden hour sunlight, 50mm f/1.8, bokeh, hyperdetailed skin, authentic look, 8k","deformed, bad anatomy, disfigured, poorly drawn face, mutation, extra limb, ugly, disgusting, blurred, watermark, bad hands, cartoon, 3d render"
"⚡ SDXL Lightning Master (8K Portrait)","Award-winning RAW cinematic portrait of an elegant woman, authentic skin pores, fine skin texture, soft chiaroscuro studio lighting, 85mm f/1.4 lens, natural gaze, luxury atmosphere, photorealistic, 8k uhd, masterpiece, <lora:add-detail-xl:0.8>","ugly, deformed, noisy, blurry, low contrast, cartoon, anime, 3d render, extra limbs, bad hands, plastic skin"
"💎 Asian Beauty Elegance","RAW close-up portrait of a gorgeous 22yo Asian woman, natural glowing skin texture, subtle makeup, traditional silk details, soft cinematic lighting, 85mm lens, 8k uhd, photorealistic","disfigured, ugly, bad art, deformed, poorly drawn, extra limbs, dull eyes, low quality"
"✨ Detail & Skin Booster","hyperdetailed skin texture, visible pores, individual eyelashes, soft rim lighting, sharp focus, 85mm portrait photography, 8k uhd, professional color grading","plastic skin, airbrushed, wax figure, smooth skin, doll, oversaturated"
""".strip()

with open("/content/stable-diffusion-webui/styles.csv", "w", encoding="utf-8") as f:
    f.write(styles_content)
print("📋 Đã cài sẵn 5 bộ Styles Prompt mẫu vào styles.csv!")

# 5. Tạo Symlink liên kết thư mục Models từ Google Drive vào Automatic1111
DRIVE_BASE = "/content/drive/MyDrive/ComfyStudio/models"
LINK_MAP = {
    f"{DRIVE_BASE}/checkpoints": "/content/stable-diffusion-webui/models/Stable-diffusion",
    f"{DRIVE_BASE}/loras": "/content/stable-diffusion-webui/models/Lora",
    f"{DRIVE_BASE}/vae": "/content/stable-diffusion-webui/models/VAE",
    f"{DRIVE_BASE}/controlnet": "/content/stable-diffusion-webui/models/ControlNet",
    f"{DRIVE_BASE}/embeddings": "/content/stable-diffusion-webui/embeddings",
    f"{DRIVE_BASE}/upscale_models": "/content/stable-diffusion-webui/models/ESRGAN"
}

for src, dst in LINK_MAP.items():
    if os.path.exists(src):
        if os.path.exists(dst) and not os.path.islink(dst):
            for item in os.listdir(dst):
                s_path = os.path.join(dst, item)
                d_path = os.path.join(src, item)
                if not os.path.exists(d_path):
                    shutil.move(s_path, d_path)
            shutil.rmtree(dst)
        if not os.path.exists(dst):
            os.symlink(src, dst)
            print(f"🔗 Symlink thành công: {src} -> {dst}")

print("\n✅ Cài đặt hoàn tất! Giao diện Lobe Theme & Trợ lý Prompt đã sẵn sàng.")


In [ ]:
#@title 🧨 3. Download Models / LoRAs (Bypass 403 Civitai & Tải đa luồng aria2c)
#@markdown Dán link Civitai (civitai.com, civitai.red), HuggingFace hoặc Direct URL.
#@markdown - *Tự động lưu vào Google Drive (`MyDrive/ComfyStudio/models`), tự nhận diện vào Automatic1111.*
#@markdown - *Cú pháp đổi tên: `link @= ten_file.safetensors`.*
#@markdown - *Hướng dẫn dùng LoRA trong A1111: gõ `<lora:tên_file_không_đuôi:1.0>` trong Positive Prompt!*

Checkpoint_Links = "" #@param {type:"string"}
Lora_Links = "" #@param {type:"string"}
Vae_Links = "" #@param {type:"string"}
Controlnet_Links = "" #@param {type:"string"}

#@markdown ---
#@markdown ### ✨ Tùy chọn Preset nhanh & Tokens:
Quick_Preset = "-- Kh\u00f4ng d\u00f9ng preset (D\u00f9ng link \u1edf tr\u00ean) --" #@param ["-- Kh\u00f4ng d\u00f9ng preset (D\u00f9ng link \u1edf tr\u00ean) --", "[Checkpoint] Realistic Vision V6.0 Hyper (2 GB)", "[LoRA] Krea 2 NSFW V4 LoRA (435 MB)", "[LoRA] Add-Detail-XL (30 MB)", "[Checkpoint] SDXL Base 1.0 (6.6 GB)", "[Checkpoint] SDXL Lightning 4-Step (6.5 GB)", "[Upscaler] 4x-UltraSharp (67 MB)", "[VAE] SDXL VAE (335 MB)"]
Civitai_API_Token = "8c7337ac0c39fe4133ae19a3d65b806f" #@param {type:"string"}
HuggingFace_Token = "" #@param {type:"string"}

import os
import re
import sys
import json
import subprocess
import urllib.request

!which aria2c > /dev/null || (apt-get update -qq && apt-get install -y -qq aria2)

DRIVE_BASE = "/content/drive/MyDrive/ComfyStudio/models"
has_drive = os.path.exists("/content/drive/MyDrive")
TARGET_BASE = DRIVE_BASE if has_drive else "/content/stable-diffusion-webui/models"

def split_name(link):
    delimiter = "@="
    if delimiter in link:
        parts = link.split(delimiter, 1)
        url = parts[0].strip()
        name = parts[1].strip()
        valid_exts = ('.safetensors', '.ckpt', '.pt', '.pth', '.bin', '.gguf')
        if not any(name.lower().endswith(ext) for ext in valid_exts):
            name = f"{name}.safetensors"
        return url, name
    return link.strip(), None

def parse_links(raw_text):
    if not raw_text:
        return []
    items = []
    for line in raw_text.strip().splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        for part in line.split(','):
            part = part.strip()
            if part:
                items.append(part)
    return items

def resolve_url_and_filename(url, civitai_token="", hf_token=""):
    url = url.strip()
    filename = None
    
    if "huggingface.co" in url:
        if "/blob/" in url:
            url = url.replace("/blob/", "/resolve/")
        if hf_token and "token=" not in url:
            sep = "&" if "?" in url else "?"
            url = f"{url}{sep}token={hf_token}"
        clean_path = url.split("?")[0]
        filename = clean_path.split("/")[-1] if "/" in clean_path else None
        return url, filename
        
    if "civitai.com" in url or "civitai.red" in url:
        m_ver = re.search(r'modelVersionId=(\d+)', url)
        vid = m_ver.group(1) if m_ver else None
        
        if not vid:
            m_api = re.search(r'/(?:api/)?download/models/(\d+)', url)
            if m_api:
                vid = m_api.group(1)
                
        if vid:
            try:
                req = urllib.request.Request(f"https://civitai.com/api/v1/model-versions/{vid}", headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=6) as r:
                    d = json.loads(r.read().decode())
                    files = d.get('files', [])
                    if files and 'name' in files[0]:
                        filename = files[0]['name']
            except Exception:
                pass
            dl_url = f"https://civitai.com/api/download/models/{vid}"
            if civitai_token:
                dl_url += f"?token={civitai_token}"
            return dl_url, filename

        m_model = re.search(r'/models/(\d+)', url)
        if m_model:
            model_id = m_model.group(1)
            try:
                req = urllib.request.Request(f"https://civitai.com/api/v1/models/{model_id}", headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=6) as r:
                    d = json.loads(r.read().decode())
                    versions = d.get('modelVersions', [])
                    if versions:
                        vid = versions[0].get('id')
                        files = versions[0].get('files', [])
                        filename = files[0].get('name') if files else None
                        dl_url = f"https://civitai.com/api/download/models/{vid}"
                        if civitai_token:
                            dl_url += f"?token={civitai_token}"
                        return dl_url, filename
            except Exception:
                pass
                
        if civitai_token and "token=" not in url:
            sep = "&" if "?" in url else "?"
            url = f"{url}{sep}token={civitai_token}"
        return url, filename

    clean_path = url.split("?")[0]
    filename = clean_path.split("/")[-1] if "/" in clean_path else None
    return url, filename

PRESET_MAP = {
    "[Checkpoint] Realistic Vision V6.0 Hyper (2 GB)": ("checkpoints", "https://civitai.com/api/download/models/501240?fileId=418901"),
    "[LoRA] Krea 2 NSFW V4 LoRA (435 MB)": ("loras", "https://civitai.com/api/download/models/3147117?fileId=3027612"),
    "[LoRA] Add-Detail-XL (30 MB)": ("loras", "https://huggingface.co/OedoSoldier/detail-tweaker-lora/resolve/main/add-detail-xl.safetensors"),
    "[Checkpoint] SDXL Base 1.0 (6.6 GB)": ("checkpoints", "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors"),
    "[Checkpoint] SDXL Lightning 4-Step (6.5 GB)": ("checkpoints", "https://huggingface.co/ByteDance/SDXL-Lightning/resolve/main/sdxl_lightning_4step.safetensors"),
    "[Upscaler] 4x-UltraSharp (67 MB)": ("upscale_models", "https://huggingface.co/lokcx/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth"),
    "[VAE] SDXL VAE (335 MB)": ("vae", "https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors"),
}

queue = []
for cat, links_text in [("checkpoints", Checkpoint_Links), ("loras", Lora_Links), ("vae", Vae_Links), ("controlnet", Controlnet_Links)]:
    for item in parse_links(links_text):
        url, name = split_name(item)
        queue.append((cat, url, name))

if Quick_Preset in PRESET_MAP:
    cat, p_url = PRESET_MAP[Quick_Preset]
    queue.append((cat, p_url, None))

if not queue:
    print("ℹ️ Chưa có link nào được nhập. Hãy dán link hoặc chọn Preset rồi chạy ô này!")
else:
    print(f"\n📋 Tổng số file cần tải: {len(queue)}")
    for idx, (cat, raw_url, explicit_name) in enumerate(queue, 1):
        dest_dir = os.path.join(TARGET_BASE, cat)
        os.makedirs(dest_dir, exist_ok=True)
        
        download_url, auto_name = resolve_url_and_filename(raw_url, Civitai_API_Token, HuggingFace_Token)
        final_name = explicit_name or auto_name
        
        if final_name:
            target_path = os.path.join(dest_dir, final_name)
            if os.path.exists(target_path) and os.path.getsize(target_path) > 1024 * 1024:
                size_gb = os.path.getsize(target_path) / (1024 ** 3)
                print(f"⏭️  [ĐÃ CÓ SẴN] {cat}/{final_name} ({size_gb:.2f} GB) -> BỎ QUA.")
                continue
        
        print(f"\n[{idx}/{len(queue)}] ⬇️ Đang tải vào {cat}/: {final_name or download_url}")
        if final_name:
            cmd = f'aria2c --console-log-level=warn --summary-interval=5 -c -x 16 -s 16 -k 1M --check-certificate=false "{download_url}" -d "{dest_dir}" -o "{final_name}"'
        else:
            cmd = f'aria2c --console-log-level=warn --summary-interval=5 -c -x 16 -s 16 -k 1M --check-certificate=false --content-disposition=true "{download_url}" -d "{dest_dir}"'
            
        res = subprocess.run(cmd, shell=True)
        if res.returncode != 0:
            print("⚠️ aria2c gặp lỗi, chuyển sang wget...")
            if final_name:
                wget_cmd = f'wget -c "{download_url}" -O "{dest_dir}/{final_name}"'
            else:
                wget_cmd = f'wget -c --content-disposition "{download_url}" -P "{dest_dir}"'
            subprocess.run(wget_cmd, shell=True)
            
        print(f"✅ Tải xong vào {cat}!")
    print("\n🎉 Hoàn thành tải model cho AUTOMATIC1111!")


In [ ]:
#@title 🚀 4. Khởi chạy AUTOMATIC1111 WebUI (Mở Link Truy cập)
#@markdown Chọn phương thức kết nối để lấy URL giao diện WebUI:

tunnel_type = "Gradio Share (Ch\u00ednh ch\u1ee7, d\u1ec5 d\u00f9ng nh\u1ea5t)" #@param ["Gradio Share (Ch\u00ednh ch\u1ee7, d\u1ec5 d\u00f9ng nh\u1ea5t)", "Ngrok Tunnel (Nhanh & \u1ed5n \u0111\u1ecbnh, c\u1ea7n Token)", "LocalTunnel (Mi\u1ec5n ph\u00ed)"]
ngrok_auth_token = "" #@param {type:"string"}
theme = "dark" #@param ["dark", "light"]

%cd /content/stable-diffusion-webui

launch_args = [
    "--theme", theme,
    "--enable-insecure-extension-access",
    "--xformers",
    "--no-half-vae",
    "--gradio-queue",
    "--outdir-txt2img-samples", "/content/drive/MyDrive/ComfyStudio/outputs/automatic1111/txt2img",
    "--outdir-img2img-samples", "/content/drive/MyDrive/ComfyStudio/outputs/automatic1111/img2img"
]

if "gradio" in tunnel_type.lower():
    launch_args.append("--share")
elif "ngrok" in tunnel_type.lower():
    if not ngrok_auth_token:
        print("⚠️ Bạn chưa nhập ngrok_auth_token! Đang tự động chuyển sang --share...")
        launch_args.append("--share")
    else:
        launch_args.extend(["--ngrok", ngrok_auth_token])

arg_string = " ".join(launch_args)
print(f"🚀 Đang khởi động WebUI với cờ: {arg_string}")
print("💡 Tìm dòng 'Running on public URL: https://...gradio.live' để nhấn vào mở giao diện WebUI!")

!python launch.py {arg_string}
